# VisDrone YOLOv8m Fine-tuning (Google Colab · T4)

Mirrors `src/training/train.py`. Run on a **free T4 GPU**. Same YOLOv8m config, hyperparameters, epochs, and DagsHub/MLflow logging as the script.

**Targets:** mAP@0.5 ≈ 0.87 in ~3 h on a T4. Report whatever the run actually produces.

## Colab run steps
1. **Set the GPU:** *Runtime → Change runtime type → Hardware accelerator: T4 GPU*.
2. **Add your DagsHub token as a Colab secret:** open the 🔑 panel (left sidebar), create a secret named **`DAGSHUB_TOKEN`** with your DagsHub access token, and enable *Notebook access*.
3. **Mount Google Drive** (the cell below) so the dataset cache, checkpoints, and `best.pt` survive runtime recycling.
4. Set your DagsHub MLflow URI + username in the **Config** cell.
5. *Runtime → Run all*.

Everything is written under a single Drive folder (`DRIVE_ROOT`) so a disconnect never loses the run; just *Run all* again to resume from the cached dataset and the last checkpoint.

In [ ]:
# --- Install deps (Colab doesn't preinstall these) ---
!pip -q install ultralytics mlflow dagshub onnx onnxruntime

In [ ]:
# --- Verify GPU (expect Tesla T4) ---
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# --- Mount Google Drive (persists dataset cache + checkpoints) ---
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/object-detection-tracking'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive project root:', DRIVE_ROOT)

In [ ]:
# --- Config: FILL THESE IN ---
import os

# DagsHub MLflow URI + username for this project (username is not secret).
MLFLOW_TRACKING_URI = 'https://dagshub.com/mohanemg07-web/object-detection-tracking.mlflow'
DAGSHUB_USERNAME = 'mohanemg07-web'

# Read the DagsHub token from Colab Secrets (never hardcode/commit it).
# os.environ is required here: a shell `!export` would NOT reach this kernel.
from google.colab import userdata
DAGSHUB_TOKEN = userdata.get('DAGSHUB_TOKEN')

os.environ['MLFLOW_TRACKING_URI'] = MLFLOW_TRACKING_URI
os.environ['MLFLOW_TRACKING_USERNAME'] = DAGSHUB_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = DAGSHUB_TOKEN or ''  # token -> MLflow password
assert DAGSHUB_TOKEN, 'Add a Colab secret named DAGSHUB_TOKEN (with notebook access).'

# All dataset + output paths live UNDER Drive (DRIVE_ROOT) so a runtime
# disconnect never forces a multi-GB re-download or loses checkpoints.
RAW_DIR = f'{DRIVE_ROOT}/data/raw'
YOLO_DIR = f'{DRIVE_ROOT}/data/yolo'
DATA_YAML = f'{DRIVE_ROOT}/data/yolo/VisDrone-DET/data.yaml'  # written by convert (below)
RUNS_DIR = f'{DRIVE_ROOT}/runs/train'   # Ultralytics per-epoch checkpoints -> Drive
WEIGHTS_DIR = f'{DRIVE_ROOT}/weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

EPOCHS = 100
IMGSZ = 640
BATCH = 16
EXPERIMENT = 'visdrone-yolov8m'

## Data preparation

Clone this repo to reuse the conversion code, then download + convert VisDrone **into the Drive folder** so it is cached across sessions. The download step is resumable; on a reconnect just re-run and it skips already-extracted archives.

In [ ]:
# --- Get the repo (replace with your fork URL) ---
# !git clone https://github.com/<you>/object-detection-tracking.git
# %cd object-detection-tracking
# !pip -q install -r requirements.txt

# Point the data pipeline at Drive via a paths override so the downloaded
# archives, the converted YOLO dataset, AND the generated data.yaml all live
# under MyDrive/object-detection-tracking (cached across runtime recycles).
import pathlib, yaml
paths = yaml.safe_load(open('configs/paths.yaml'))
paths['raw_dir'] = RAW_DIR        # MyDrive/.../data/raw
paths['yolo_dir'] = YOLO_DIR      # MyDrive/.../data/yolo
paths['data_yaml'] = DATA_YAML    # MyDrive/.../data/yolo/VisDrone-DET/data.yaml
pathlib.Path('configs/paths.colab.yaml').write_text(yaml.safe_dump(paths))
print('raw  ->', paths['raw_dir'])
print('yolo ->', paths['yolo_dir'])
print('yaml ->', paths['data_yaml'])

# Resumable: re-running after a disconnect skips already-extracted archives.
!python -m src.data.download_visdrone --config configs/paths.colab.yaml
!python -m src.data.convert_visdrone  --config configs/paths.colab.yaml --write-data-yaml
!python -m src.data.validate_labels   --config configs/paths.colab.yaml

In [ ]:
# --- Train with MLflow logging to DagsHub ---
import mlflow
from ultralytics import YOLO

if MLFLOW_TRACKING_URI:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(EXPERIMENT)

model = YOLO('yolov8m.pt')

def _log_epoch(trainer):
    if not MLFLOW_TRACKING_URI:
        return
    m = {k.replace('(', '').replace(')', ''): float(v) for k, v in trainer.metrics.items()}
    mlflow.log_metrics(m, step=trainer.epoch)

model.add_callback('on_fit_epoch_end', _log_epoch)

run = mlflow.start_run() if MLFLOW_TRACKING_URI else None
results = model.train(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    device=0, seed=42, cos_lr=True, close_mosaic=10,
    project=RUNS_DIR, name='yolov8m_visdrone',  # checkpoints persisted to Drive
)
print('Saved to:', results.save_dir)

In [ ]:
# --- Log final metrics + best.pt as artifacts, copy best.pt to Drive, close run ---
import shutil
from pathlib import Path

save_dir = Path(results.save_dir)
best = save_dir / 'weights' / 'best.pt'

if MLFLOW_TRACKING_URI:
    if model.metrics and model.metrics.results_dict:
        mlflow.log_metrics({k.replace('(', '').replace(')', ''): float(v)
                            for k, v in model.metrics.results_dict.items()
                            if isinstance(v, (int, float))})
    for p in ['confusion_matrix.png', 'results.png', 'PR_curve.png']:
        fp = save_dir / p
        if fp.exists():
            mlflow.log_artifact(str(fp), 'plots')
    if best.exists():
        mlflow.log_artifact(str(best), 'weights')
    mlflow.end_run()

# Persist best.pt to a stable Drive location so a disconnect never loses it.
if best.exists():
    dest = Path(WEIGHTS_DIR) / 'best.pt'
    shutil.copy2(best, dest)
    print('best.pt copied to Drive:', dest)
print('best.pt:', best, '| exists:', best.exists())

## After training

`best.pt` is on Drive at `MyDrive/object-detection-tracking/weights/best.pt`. Download it and place it at `weights/best.pt` in your local repo, then resume with **Phase 4 (ONNX export + INT8 quantization)**. Paste the real mAP@0.5 / mAP@0.5:0.95 into the README results table.